# 01 · ZIPA-CLI Quickstart

This notebook walks through installing, listing models, and running phonetic
decoding on a single file and a directory. It builds a *tiny* demo ONNX model so it
runs end-to-end without large downloads; for real work, replace the model path with
a registry tag such as `zipa-cr-s-300k`.

**Install** (minimal ONNX backend):
```bash
pip install -e "..[onnx]"   # from the repo root
```

In [ ]:
# Make the zipa_cli package importable when running from tutorials/ without install.
import sys, os
REPO = os.path.abspath('..')
if REPO not in sys.path: sys.path.insert(0, REPO)
from zipa_cli.cli import main as zipa  # call like zipa(['models','list'])
print('zipa-cli importable from', REPO)

In [ ]:
# Which backend deps are available in this kernel?
def have(mod):
    try:
        __import__(mod); return True
    except Exception:
        return False
ONNX_OK = all(have(m) for m in ['onnxruntime','soundfile','lhotse'])
print('ONNX inference available:', ONNX_OK)
print('(install with: pip install -e "..[onnx]")')

## Browse the model registry
Registry commands need no heavy deps.

In [ ]:
zipa(['models','list'])

In [ ]:
zipa(['models','info','zipa-cr-s-300k'])

> Downloading a real model: `zipa(['models','download','zipa-cr-s-300k','--backend','onnx'])`

In [ ]:
# Build a tiny *valid* ONNX CTC model + tokens so this notebook runs end-to-end
# WITHOUT downloading the multi-hundred-MB ZIPA checkpoints. Real usage just swaps
# --model <path> for a registry tag like 'zipa-cr-s-300k'.
import numpy as np, tempfile
TMP = tempfile.mkdtemp(prefix='zipa_demo_')
def build_tiny_ctc():
    import onnx
    from onnx import helper, TensorProto
    V = 10
    W = (np.random.RandomState(0).randn(80, V)*0.1).astype(np.float32)
    g = helper.make_graph([helper.make_node('MatMul',['x','W'],['logits'])],'tiny',
        [helper.make_tensor_value_info('x',TensorProto.FLOAT,['B','T',80]),
         helper.make_tensor_value_info('x_lens',TensorProto.INT64,['B'])],
        [helper.make_tensor_value_info('logits',TensorProto.FLOAT,['B','T',V])],
        [helper.make_tensor('W',TensorProto.FLOAT,[80,V],W.flatten())])
    m = helper.make_model(g, opset_imports=[helper.make_opsetid('',13)]); m.ir_version=9
    p = os.path.join(TMP,'tiny_ctc.onnx'); onnx.save(m,p)
    tk = os.path.join(TMP,'tokens.txt')
    with open(tk,'w') as f:
        for i,t in enumerate('<blk> a b c d e f g h i'.split()): f.write(f'{t} {i}\n')
    return p, tk
MODEL = TOKENS = None
if ONNX_OK and have('onnx'):
    MODEL, TOKENS = build_tiny_ctc()
    print('demo model:', MODEL)
else:
    print('Skipping demo model (need onnxruntime+lhotse+onnx). Markdown cells still apply.')

## Transcribe a single file
We use the bundled `zipa/sample.wav`. With a real model you would write
`zipa(['transcribe','sample.wav','--model','zipa-cr-s-300k'])`.

In [ ]:
SAMPLE = os.path.join(REPO,'zipa','sample.wav')
if MODEL:
    zipa(['transcribe', SAMPLE, '--model', MODEL, '--model-type','ctc','--tokens',TOKENS])
else:
    print('install onnx extras to run; expected output is a space-separated phone string')

## Batch-decode a directory to TSV

In [ ]:
import shutil
AUDIO_DIR = os.path.join(TMP,'audio'); os.makedirs(AUDIO_DIR, exist_ok=True)
if MODEL:
    for n in ['a.wav','b.wav']: shutil.copy(SAMPLE, os.path.join(AUDIO_DIR,n))
    out = os.path.join(TMP,'out.tsv')
    zipa(['decode','--input',AUDIO_DIR,'--input-type','dir','--model',MODEL,
          '--model-type','ctc','--tokens',TOKENS,'-o',out,'--batch-size','2'])
    print('---- out.tsv ----'); print(open(out).read())

### Real-world equivalent
```bash
zipa-cli decode --input audio_dir/ --input-type dir \
    --model zipa-cr-l-500k --max-duration 600 -o transcripts.tsv
```